In [11]:
# Импортируем базовые библиотеки для анализа данных и построения графиков
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Импортируем инструменты машинного обучения: разделение выборки, модель регрессии и генератор полиномиальных признаков
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# Импортируем метрики для оценки качества регрессионных моделей
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [12]:
# Загружаем скачанный файл датасета с явным указанием кодировки 'latin1' для предотвращения ошибок чтения
df = pd.read_csv('Sample - Superstore.csv', encoding='latin1')
# Исследуем размерность таблицы, типы данных колонок и основные описательные статистики
print("Размерность данных:", df.shape)
print("\nТипы данных колонок:")
print(df.dtypes)
print("\nОписательная статистика:")
print(df.describe())

# Выбираем числовые признаки для построения предсказаний (Sales — целевая переменная)
features = ['Quantity', 'Discount', 'Profit']

# Проверяем выбранные признаки на наличие пропущенных значений (NaN)
print("\nКоличество пропусков в признаках:")
print(df[features].isnull().sum())

FileNotFoundError: [Errno 2] No such file or directory: 'Sample - Superstore.csv'

In [ ]:
# Импортируем библиотеку seaborn для красивой визуализации данных
import seaborn as sns

# Строим матрицу корреляции Пирсона между числовыми предикторами и целевой переменной Sales
corr = df[['Sales', 'Quantity', 'Discount', 'Profit']].corr()

# Визуализируем полученную матрицу в виде тепловой карты с отображением значений и цветовой схемой coolwarm
sns.heatmap(corr, annot=True, cmap='coolwarm')

# Добавляем заголовок к графику и выводим его на экран
plt.title('Feature Correlations with Sales')
plt.show()

In [ ]:
# Выделяем матрицу признаков X (предикторы) и вектор целевой переменной y (Sales)
X = df[['Quantity', 'Discount', 'Profit']]
y = df['Sales']

# Разделяем данные на обучающий и тестовый наборы в пропорции 80% на 20%
# random_state=42 гарантирует воспроизводимость результатов при повторных запусках
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Импортируем класс линейной регрессии
from sklearn.linear_model import LinearRegression

# Инициализируем модель линейной регрессии
model = LinearRegression()

# Обучаем модель на тренировочных данных
model.fit(X_train, y_train)

# Получаем прогнозы модели на тестовой выборке для последующей оценки качества
y_pred = model.predict(X_test)

In [ ]:
# Импортируем функции для расчета основных метрик регрессии
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Вычисляем среднеквадратичную ошибку (MSE)
mse = mean_squared_error(y_test, y_pred)

# Вычисляем корень из среднеквадратичной ошибки (RMSE)
rmse = np.sqrt(mse)

# Вычисляем среднюю абсолютную ошибку (MAE)
mae = mean_absolute_error(y_test, y_pred)

# Вычисляем коэффициент детерминации (R²), показывающий долю объясненной дисперсии
r2 = r2_score(y_test, y_pred)

# Выводим все метрики на экран с форматированием до 2 и 4 знаков после запятой
print(f"MSE: {mse:.2f}, RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.4f}")

In [ ]:
# Проходим циклом по названиям признаков и их соответствующим весам (коэффициентам) модели
for feature, coef in zip(X.columns, model.coef_):
    print(f"{feature}: {coef:.4f}")

# Выводим свободный член (сдвиг) линейного уравнения регрессии
print(f"Intercept: {model.intercept_:.4f}")

Step 2 — Business Interpretation:

A one-unit increase in Quantity is associated with a $X (insert Quantity coefficient) increase in Sales, holding other features constant.

A one-unit increase in Discount is associated with a $Y (insert Discount coefficient) change in Sales, holding other features constant.

Step 3 — Identifying Unexpected Signs:

Check the sign of the Discount coefficient. In many retail datasets, it might unexpectedly turn out to be negative. A potential hypothesis for this is that aggressive discounting heavily slashes the gross revenue generated per item, or that deep discounts are primarily applied to low-value clearance products to clear out dead stock.

In [ ]:
# Import PolynomialFeatures from sklearn's preprocessing module
from sklearn.preprocessing import PolynomialFeatures

# Step 1: Create polynomial features (degree 2), excluding the bias column
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# Step 2: Train a new linear regression model on the generated polynomial features
model_poly = LinearRegression()
model_poly.fit(X_train_poly, y_train)
y_pred_poly = model_poly.predict(X_test_poly)

# Step 3: Compute evaluation metrics for the polynomial model
mse_poly = mean_squared_error(y_test, y_pred_poly)
rmse_poly = np.sqrt(mse_poly)
mae_poly = mean_absolute_error(y_test, y_pred_poly)
r2_poly = r2_score(y_test, y_pred_poly)

# Print performance comparisons side by side
print("--- Linear Model vs. Polynomial Model ---")
print(f"MSE:  Linear = {mse:.2f}  | Poly = {mse_poly:.2f}")
print(f"RMSE: Linear = {rmse:.2f}  | Poly = {rmse_poly:.2f}")
print(f"MAE:  Linear = {mae:.2f}  | Poly = {mae_poly:.2f}")
print(f"R2:   Linear = {r2:.4f}  | Poly = {r2_poly:.4f}")

Option B: If Polynomial Features Hurt or Stayed the Same (Higher/similar errors)
Step 4 — Evaluation Analysis:
Incorporating polynomial features did not yield a notable performance lift and instead risked overfitting the training subset. The baseline linear regression model remains preferable because it maintains simpler structural complexity and higher interpretability while delivering comparable metrics, suggesting that the underlying patterns in this subset of features are adequately modeled linearly.

In [ ]:
# Set up the figure size for the first diagnostic plot
plt.figure(figsize=(8, 6))

# Create a scatter plot comparing actual sales values with the model's predictions
# alpha=0.5 makes overlapping points semi-transparent, s=20 scales the point size
plt.scatter(y_test, y_pred, alpha=0.5, s=20)

# Draw a red dashed diagonal line (y = x) representing perfect predictions
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)

# Set labels and title for clear visualization
plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.title('Predicted vs Actual Sales')

# Display the plot
plt.show()

In [ ]:
# Calculate the prediction errors (residuals) by subtracting predictions from actuals
residuals = y_test - y_pred

# Set up the figure size for the residual analysis plot
plt.figure(figsize=(8, 6))

# Create a scatter plot of predictions vs. their corresponding residual errors
plt.scatter(y_pred, residuals, alpha=0.5, s=20)

# Add a horizontal reference line at 0; points above are under-predictions, below are over-predictions
plt.axhline(y=0, color='r', linestyle='--')

# Set labels and title
plt.xlabel('Predicted Sales')
plt.ylabel('Residuals')
plt.title('Residual Plot')

# Display the plot
plt.show()

Step 3 — Diagnostic Plots Analysis:Predicted vs. Actual Plot: Look at how closely your data points hug the red dashed line. If they form a tight cluster straight along the diagonal, your model has strong predictive accuracy. If points spread out wildly like a cloud, the model struggles with high-value transactions.Residual Plot: Check the distribution of points around the center line ($y=0$).Random Scatter (Good): If the dots are randomly distributed above and below the line with no distinct shape, it means your linear model's assumptions hold true.Systematic Patterns (Bad): If you see a clear shape (like a funnel expanding outwards, or a curved U-shape), it means your model is missing crucial non-linear relationships or suffering from heteroscedasticity. In this retail dataset, you will likely notice a funnel shape because large transactions naturally create larger absolute pricing errors.